# Ir Além 2 — Análise dos resultados do robô de monitoramento

Este notebook lê o que o robô de RPA (`src/robo_monitoramento.py`) já gravou
no SQLite e no MongoDB e gera uma visão executiva dos resultados — no mesmo
espírito da etapa de visualização apresentada na aula (scatterplot e
barplot sobre os dados persistidos pela automação).

**Pré-requisito:** rode `python src/gerar_dados_simulados.py` e depois
`python src/robo_monitoramento.py --once` pelo menos uma vez antes deste
notebook, para que existam dados a analisar.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../src"))

import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pymongo import MongoClient

import config

sns.set_theme(style="whitegrid")

## 1. Sinais vitais avaliados (SQLite)

In [ ]:
conn = sqlite3.connect(config.SQLITE_PATH)
df_sinais = pd.read_sql("""
    SELECT sv.*, p.nome
    FROM sinais_vitais sv
    JOIN pacientes p ON p.id = sv.paciente_id
""", conn)
df_sinais.tail(15)

In [ ]:
print("Total de leituras:", len(df_sinais))
print("Já avaliadas pelo robô:", (df_sinais['processado_robo'] == 1).sum())
print("Marcadas por regra fixa:", (df_sinais['alerta_regra'] == 1).sum())
print("Marcadas como anomalia pelo IsolationForest:", (df_sinais['anomalia_ia'] == 1).sum())

## 2. Dispersão pressão x frequência cardíaca

Cada ponto é uma leitura. Cor = rótulo do IsolationForest; forma = se
disparou alguma regra fixa. O objetivo é visualizar que os dois métodos
capturam casos diferentes: regras fixas pegam valores extremos isolados,
o IsolationForest pega combinações atípicas mesmo dentro da faixa
"aceitável" de cada variável isolada.

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(
    data=df_sinais,
    x="pressao_sistolica", y="frequencia_cardiaca",
    hue="anomalia_ia", style="alerta_regra",
    palette={0: "#4C72B0", 1: "#C44E52", None: "#999999"},
    s=70,
)
plt.title("Sinais vitais: pressão sistólica x frequência cardíaca")
plt.xlabel("Pressão sistólica (mmHg)")
plt.ylabel("Frequência cardíaca (bpm)")
plt.legend(title="anomalia_ia / alerta_regra", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 3. Alertas registrados (MongoDB) — rastreabilidade

In [ ]:
cliente = MongoClient(config.MONGO_URI)
mongo_db = cliente[config.MONGO_DB_NAME]

df_alertas = pd.DataFrame(list(mongo_db["alertas"].find()))
df_alertas.head(10)

### 3.1 Alertas por paciente e por origem

Reproduz, adaptado, o gráfico de barras da aula (taxa de anomalia por
robô) — aqui, quantidade de alertas por paciente, separados pela técnica
que os detectou (regra fixa, IsolationForest ou interpretação de texto).

In [ ]:
if not df_alertas.empty:
    plt.figure(figsize=(7, 5))
    sns.countplot(data=df_alertas, x="paciente_id", hue="origem")
    plt.title("Alertas gerados por paciente e por origem")
    plt.xlabel("ID do paciente")
    plt.ylabel("Nº de alertas")
    plt.tight_layout()
    plt.show()
else:
    print("Nenhum alerta registrado ainda — rode o robô pelo menos uma vez.")

## 4. Histórico de execuções do robô (log de auditoria)

In [ ]:
df_logs = pd.DataFrame(list(mongo_db["logs_execucao"].find())).sort_values("inicio", ascending=False)
colunas = ["execucao_id", "inicio", "duracao_segundos", "leituras_avaliadas",
           "mensagens_avaliadas", "alertas_gerados", "status"]
df_logs[colunas].head(10)

## 5. Conclusão

As duas visualizações acima resumem a rastreabilidade do pipeline: é
possível, a partir de um único notebook, (1) ver o estado bruto dos sinais
vitais e seus rótulos no banco relacional, e (2) auditar cada alerta —
inclusive os originados de mensagens de texto — até a execução exata do
robô que o gerou, no banco não relacional. Essa separação (dado
estruturado de fonte de verdade no SQLite, trilha de auditoria e conteúdo
textual variável no MongoDB) é a decisão de projeto central discutida no
relatório técnico.